In [ ]:
 !pip uninstall -y httpx
!pip install httpx==0.27.0
!pip install -U sentence-transformers transformers

Found existing installation: httpx 0.28.1
Uninstalling httpx-0.28.1:
  Successfully uninstalled httpx-0.28.1


In [ ]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ======================
# CONFIG
# ======================

DATASET_PATH = "/content/phi4_hindi.csv"
OUTPUT_FILE = "phase1_results_hin_phi4.csv"

# ======================

print("📂 Loading dataset...")
df = pd.read_csv(DATASET_PATH)
print("Columns:", df.columns)

# Auto detect columns
def detect(names):
    for c in df.columns:
        for n in names:
            if n.lower() in c.lower():
                return c
    return None

Q_COL = detect(["question"])
GEN_COL = detect(["generated", "answer"])
REF_COL = detect(["reference"])

print("Using:", Q_COL, GEN_COL, REF_COL)

# ======================
# MODELS
# ======================

print("🔤 Loading multilingual similarity model...")
sim_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("🌍 Loading NLLB translation model...")
translator_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
translator_model = translator_model.to(DEVICE)

# ======================
# FUNCTIONS
# ======================

def sim(a, b):
    emb = sim_model.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

def repetition(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return 1 - (len(set(words)) / len(words))

def length_ratio(g, r):
    return len(g.split()) / (len(r.split()) + 1e-5)

def drift(q, g):
    return sim(q, g)

def back_translate_nllb(text, src_lang="hin_Deva", tgt_lang="eng_Latn"):
    try:
        inputs = translator_tokenizer(text, return_tensors="pt", truncation=True).to(DEVICE)

        translated_tokens = translator_model.generate(
            **inputs,
            forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids(tgt_lang),
            max_length=200
        )

        return translator_tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

    except Exception as e:
        return text

# ======================
# MAIN LOOP
# ======================

results = []

print("🔬 Running Phase 1 evaluation...")

for _, row in tqdm(df.iterrows(), total=len(df)):

    q = str(row[Q_COL])
    g = str(row[GEN_COL])
    r = str(row[REF_COL])

    if len(g.strip()) == 0:
        continue

    # Back translation (Bengali → English)
    g_en = back_translate_nllb(g)

    # Similarities
    sim_multi = sim(g, r)        # cross-lingual
    sim_trans = sim(g_en, r)     # translated

    # Other metrics
    d = drift(q, g)
    rep = repetition(g)
    lr = length_ratio(g, r)

    # Keep full original row
    result_row = row.to_dict()

    result_row.update({
        "generated_translated_en": g_en,
        "similarity_multilingual": sim_multi,
        "similarity_translated": sim_trans,
        "hallucinated": sim_trans < 0.6,
        "drift_score": d,
        "repetition_score": rep,
        "length_ratio": lr
    })

    results.append(result_row)

# ======================
# SAVE
# ======================

pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

print("✅ Phase 1 complete →", OUTPUT_FILE)

📂 Loading dataset...
Columns: Index(['id', 'model', 'language', 'question_en', 'question_hi', 'reference_en',
       'answer_generated_hi'],
      dtype='object')
Using: question_en answer_generated_hi reference_en
🔤 Loading multilingual similarity model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

🔬 Running Phase 1 evaluation...



100%|██████████| 817/817 [54:39<00:00,  4.01s/it]

✅ Phase 1 complete → phase1_results_hin_phi4.csv
